In [56]:
#IMPORTAÇÕES:
import pandas as pd
import numpy as np

#CARREGAMENTO DOS DADOS:
arquivo= 'KPMG_VI_New_raw_data_update_final.xlsx'
df = pd.read_excel(arquivo, sheet_name= 'Transactions', header=1)

In [57]:
#CONFERENCIA DE CABEÇALHO E DOS TIPOS:
df.head()
#Precisei colocar [sheet_name= 'Transactions', header=1] na df para exibir corretamente.

,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
0,1,2,2950,2017-02-25,0.0,Approved,Solex,Standard,medium,medium,71.49,53.62,41245.0
1,2,3,3120,2017-05-21,1.0,Approved,Trek Bicycles,Standard,medium,large,2091.47,388.92,41701.0
2,3,37,402,2017-10-16,0.0,Approved,OHM Cycles,Standard,low,medium,1793.43,248.82,36361.0
3,4,88,3135,2017-08-31,0.0,Approved,Norco Bicycles,Standard,medium,medium,1198.46,381.10,36145.0
4,5,78,787,2017-10-01,1.0,Approved,Giant Bicycles,Standard,medium,large,1765.30,709.48,42226.0


In [58]:
#DUPLICATAS EXATAS:
for col in df.columns:
    print(f"{col}: {df[col].duplicated().sum()}")
#Sim, existe

transaction_id: 0
product_id: 19899
customer_id: 16506
transaction_date: 19636
online_order: 19997
order_status: 19998
brand: 19993
product_line: 19995
product_class: 19996
product_size: 19996
list_price: 19704
standard_cost: 19896
product_first_sold_date: 19899


In [59]:
#DUPLICATAS POR SUBCONUNTO DE COLUNAS:
subconjunto = ['customer_id', 'transaction_date', 'product_id']

total_duplicadas = df.duplicated(subset=subconjunto).sum()
print(total_duplicadas)

#Juntas, essas três colunas formam uma chave natural robusta para mapear transações ou itens de transação quando o ID principal está ausente.

1


In [60]:
#UNICIDADE DOS IDENTIFICADORES:
colunas_id = ['transaction_id', 'customer_id', 'product_id']

for col in colunas_id:
    if col in df.columns:
        total_linhas = len(df)
        total_unicos = df[col].nunique()
        nulos = df[col].isnull().sum()
        is_unique = df[col].is_unique
        
        print(f"Coluna: {col}")
        print(f"   - Total de linhas: {total_linhas}")
        print(f"   - Valores únicos (.nunique()): {total_unicos}")
        print(f"   - Valores nulos/ausentes: {nulos}")
        print(f"   - É 100% única? (.is_unique) {is_unique}")
        print("-" * 45)

Coluna: transaction_id
   - Total de linhas: 20000
   - Valores únicos (.nunique()): 20000
   - Valores nulos/ausentes: 0
   - É 100% única? (.is_unique) True
---------------------------------------------
Coluna: customer_id
   - Total de linhas: 20000
   - Valores únicos (.nunique()): 3494
   - Valores nulos/ausentes: 0
   - É 100% única? (.is_unique) False
---------------------------------------------
Coluna: product_id
   - Total de linhas: 20000
   - Valores únicos (.nunique()): 101
   - Valores nulos/ausentes: 0
   - É 100% única? (.is_unique) False
---------------------------------------------


In [ ]:
#ESTRATÉGIA DE TRATAMENTO E IMPACTO:

"""
A remoção de duplicatas por chave composta (keep='first'): A estratégia consiste em utilizar um 
subconjunto natural de colunas (customer_id, transaction_date, product_id) para identificar registros 
redundantes e manter apenas a primeira ocorrência de cada transação, eliminando as cópias excedentes.

Porque:
-Linhas duplicadas inflacionam artificialmente indicadores cruciais (como a soma de list_price), 
gerando relatórios de faturamento distorcidos. A remoção restaura a precisão dos números;

-Como o transaction_id original apresentou falhas/ausência de dados, a chave composta atua como um 
identificador confiável para capturar repetições exatas geradas por falhas de sistema ou importação;

-Manter o primeiro registro (keep='first') garante que dados válidos não sejam perdidos, mantendo a 
integridade histórica da base sem prejudicar análises futuras.
"""

# 1. Definir o subconjunto de colunas para identificar duplicatas
subconjunto = ['customer_id', 'transaction_date', 'product_id']

# 2. Medir o indicador antes de qualquer alteração
soma_preco_antes = df['list_price'].sum()
total_linhas_antes = len(df)

# 3. Identificar e remover duplicatas (mantendo a primeira ocorrência)
# Criamos um DataFrame limpo
df_limpo = df.drop_duplicates(subset=subconjunto, keep='first').copy()

# 4. Calcular registros afetados
total_linhas_depois = len(df_limpo)
registros_removidos = total_linhas_antes - total_linhas_depois

# 5. Medir o indicador após a remoção
soma_preco_depois = df_limpo['list_price'].sum()
impacto_absoluto = soma_preco_antes - soma_preco_depois
impacto_percentual = (impacto_absoluto / soma_preco_antes) * 100

# 6. Exibir os resultados da auditoria
print(f"--- RELATÓRIO DE IMPACTO DA LIMPEZA ---")
print(f"• Total de registros antes: {total_linhas_antes}")
print(f"• Total de registros depois: {total_linhas_depois}")
print(f"• Registros duplicados removidos (afetados): {registros_removidos}")
print(f"----------------------------------------")
print(f"• Soma de 'list_price' ANTES: R$ {soma_preco_antes:,.2f}")
print(f"• Soma de 'list_price' DEPOIS: R$ {soma_preco_depois:,.2f}")
print(f"• Impacto absoluto na receita (list_price): R$ {impacto_absoluto:,.2f}")
print(f"• Impacto percentual: {impacto_percentual:.2f}%")

--- RELATÓRIO DE IMPACTO DA LIMPEZA ---
• Total de registros antes: 20000
• Total de registros depois: 19999
• Registros duplicados removidos (afetados): 1
----------------------------------------
• Soma de 'list_price' ANTES: R$ 22,156,588.98
• Soma de 'list_price' DEPOIS: R$ 22,156,386.36
• Impacto absoluto na receita (list_price): R$ 202.62
• Impacto percentual: 0.00%


In [ ]:
#RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO:
""" Não, porque um dado pode ter se repetido como 'Andreas' e 'andreas' dependendo do codigo"""